# Data Scraping Ulasan Aplikasi HOK

In [1]:
!pip install google-play-scraper transformers torch scikit-learn pandas matplotlib seaborn pipreqs

In [13]:
!pipreqs "/content" --scan-notebooks

INFO: Successfully saved requirements file in /content/requirements.txt


## Library Preparation

In [2]:
from google_play_scraper import reviews, Sort
import pandas as pd
import re
import torch
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
import nltk
nltk.download('stopwords')
from nltk.corpus import stopwords

stop_words_id = set(stopwords.words('indonesian'))

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


## Scraping 10000 ulasan

In [3]:
result, continuation_token = reviews(
    'com.levelinfinite.sgameGlobal',
    lang='id',
    country='id',
    sort=Sort.NEWEST,
    count=10000
)

In [4]:
df = pd.DataFrame(result)
df.to_csv("ulasan_hok.csv", index=False)
print("Total data mentah:", len(df))

Total data mentah: 10000


# IndoBERT OPTIMIZED 90:10

## Preprocessing data

In [5]:
df = pd.read_csv('ulasan_hok.csv')

def label_sentiment(score):
    if score >= 4:
        return "positif"
    elif score == 3:
        return "netral"
    else:
        return "negatif"

def reduce_repetitive_chars(text):
    return re.sub(r'(.)\1+', r'\1', text)

def clean_text(text):
    # Ensure text is a string before proceeding
    if not isinstance(text, str):
        return ""

    text = re.sub(r'@[A-Za-z0-9]+', '', text)
    text = re.sub(r'#', '', text)
    text = re.sub(r'RT[\s]+', '', text)
    text = re.sub(r'https?://\S+', '', text)
    text = re.sub(r',', '', text)
    text = reduce_repetitive_chars(text)
    text = re.sub(r'[^a-zA-Z ]', '', text)
    text = text.lower()
    # Stopword removal
    words = text.split()
    filtered_words = [word for word in words if word not in stop_words_id]
    text = ' '.join(filtered_words)
    return text

df['cleaned_text'] = df['content'].apply(clean_text)
df["label"] = df["score"].apply(label_sentiment)

print(df[['content', 'cleaned_text', 'label']].head(10))

                                             content  \
0  kenapa tidak bisa live streaming di tik tok ap...   
1            seruuu sekali aku pernah menang 20 kali   
2  ini kenapa sih kok gak bisa log in udah ngikut...   
3                       tolong perbaiki bug download   
4  bye TIMI gw pensi kalah terus kecewa gw maen h...   
5  selama main party kok aku sering dapat role ya...   
6  Minn game hok seru tapi map Nya bosenin gitu d...   
7  game burik gerafik burik aitem burik apalah ga...   
8  pindah dari ml krna id ml playernya bikin saki...   
9                                        terbaik lah   

                                        cleaned_text    label  
0             live streaming tik tok aplikasi ya aov  negatif  
1                                   seru menang kali  positif  
2  sih gak log in udah ngikutin saran aja gak log in  negatif  
3                       tolong perbaiki bug download  negatif  
4  bye timi gw pensi kalah kecewa gw maen hok men...  negatif  

In [6]:
print("Updated label distribution:")
df["label"].value_counts()

Updated label distribution:


,count
label,
positif,6289
negatif,3100
netral,611


## Pelabelan data

In [7]:
from imblearn.over_sampling import RandomOverSampler

# Ensure 'label_id' is present after cleaning and reloading
label_map = {"positif": 0, "netral": 1, "negatif": 2}
df['label_id'] = df['label'].map(label_map)

# Prepare data for oversampling
X = df[['cleaned_text']]
y = df['label_id']

# Apply RandomOverSampler
ros = RandomOverSampler(random_state=42)
X_resampled, y_resampled = ros.fit_resample(X, y)

# Create a new DataFrame with resampled data
df_resampled = pd.DataFrame(X_resampled, columns=['cleaned_text'])
df_resampled['label_id'] = y_resampled

# Map label_id back to original labels for verification
inverse_label_map = {v: k for k, v in label_map.items()}
df_resampled['label'] = df_resampled['label_id'].map(inverse_label_map)

print("Label distribution after RandomOverSampler:")
print(df_resampled['label'].value_counts())

Label distribution after RandomOverSampler:
label
negatif    6289
positif    6289
netral     6289
Name: count, dtype: int64


## Train Model

In [8]:
from sklearn.model_selection import train_test_split
from transformers import BertTokenizer, BertForSequenceClassification, Trainer, TrainingArguments

model_name = "indobenchmark/indobert-base-p2"
tokenizer = BertTokenizer.from_pretrained(model_name)

# Split Data 90:10 from the resampled dataframe
train_texts_resampled, test_texts_resampled, train_labels_resampled, test_labels_resampled = train_test_split(
    df_resampled['cleaned_text'].tolist(), df_resampled['label_id'].tolist(), test_size=0.1, random_state=42
)

# Tokenization
train_enc_resampled = tokenizer(train_texts_resampled, truncation=True, padding=True, max_length=128)
test_enc_resampled = tokenizer(test_texts_resampled, truncation=True, padding=True, max_length=128)

class Dataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels
    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item
    def __len__(self):
        return len(self.labels)

# Training with tuned hyperparameters
model_tuned = BertForSequenceClassification.from_pretrained(model_name, num_labels=3)

training_args_tuned = TrainingArguments(
    output_dir='./results_tuned',
    num_train_epochs=7, # Tuned epoch (from 5 to 7)
    learning_rate=5e-5, # Tuned learning rate (from 2e-5 to 5e-5)
    per_device_train_batch_size=16,
    logging_dir='./logs_tuned',
    logging_steps=500
)

trainer_tuned = Trainer(
    model=model_tuned,
    args=training_args_tuned,
    train_dataset=Dataset(train_enc_resampled, train_labels_resampled),
    eval_dataset=Dataset(test_enc_resampled, test_labels_resampled)
)

trainer_tuned.train()


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 3


wandb: You chose "Don't visualize my results"


Step,Training Loss
500,0.736000
1000,0.496100
1500,0.335000
2000,0.290900
2500,0.235300
3000,0.211500
3500,0.179600
4000,0.161900
4500,0.125100
5000,0.107600


TrainOutput(global_step=7434, training_loss=0.21689060756138392, metrics={'train_runtime': 2778.7092, 'train_samples_per_second': 42.775, 'train_steps_per_second': 2.675, 'total_flos': 7635171101355000.0, 'train_loss': 0.21689060756138392, 'epoch': 7.0})

In [9]:
predictions_output_tuned = trainer_tuned.predict(Dataset(test_enc_resampled, test_labels_resampled))

predictions_tuned = predictions_output_tuned.predictions
true_labels_tuned = predictions_output_tuned.label_ids

predicted_labels_tuned = predictions_tuned.argmax(axis=1)

print("Predicted Labels (first 10):", predicted_labels_tuned[:10])
print("True Labels (first 10):", true_labels_tuned[:10])

print("\nClassification Report for Tuned IndoBERT Model:")
print(classification_report(true_labels_tuned, predicted_labels_tuned, target_names=list(label_map.keys())))
print(f"Accuracy Score for Tuned IndoBERT Model: {accuracy_score(true_labels_tuned, predicted_labels_tuned):.4f}")

Predicted Labels (first 10): [2 0 2 2 0 1 2 2 1 1]
True Labels (first 10): [2 0 2 2 0 1 2 2 2 1]

Classification Report for Tuned IndoBERT Model:
              precision    recall  f1-score   support

     positif       0.90      0.90      0.90       614
      netral       0.97      0.97      0.97       658
     negatif       0.92      0.92      0.92       615

    accuracy                           0.93      1887
   macro avg       0.93      0.93      0.93      1887
weighted avg       0.93      0.93      0.93      1887

Accuracy Score for Tuned IndoBERT Model: 0.9306


## Inference on Custom Text

In [15]:
def predict_sentiment(text):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model_tuned.to(device)

    cleaned_text = clean_text(text)
    inputs = tokenizer(cleaned_text, return_tensors="pt", truncation=True, padding=True, max_length=128)
    inputs = {key: val.to(device) for key, val in inputs.items()}

    with torch.no_grad():
        outputs = model_tuned(**inputs)
    logits = outputs.logits
    prediction = torch.argmax(logits, dim=-1).item()

    # Inverse map to get the label name
    sentiment_label = inverse_label_map[prediction]
    return sentiment_label

# Example usage
text_to_predict = "Game ini sangat seru dan tidak pernah lag, saya suka sekali!"
sentiment = predict_sentiment(text_to_predict)
print(f"The sentiment of the text '{text_to_predict}' is: {sentiment}")

text_to_predict_2 = "Grafiknya sangat buruk dan banyak bug, saya kecewa."
sentiment_2 = predict_sentiment(text_to_predict_2)
print(f"The sentiment of the text '{text_to_predict_2}' is: {sentiment_2}")

text_to_predict_3 = "Game ini biasa saja, tidak ada yang istimewa."
sentiment_3 = predict_sentiment(text_to_predict_3)
print(f"The sentiment of the text '{text_to_predict_3}' is: {sentiment_3}")

The sentiment of the text 'Game ini sangat seru dan tidak pernah lag, saya suka sekali!' is: positif
The sentiment of the text 'Grafiknya sangat buruk dan banyak bug, saya kecewa.' is: negatif
The sentiment of the text 'Game ini biasa saja, tidak ada yang istimewa.' is: positif
